# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is accessed via a Croissant schema URL. All references to record sets, fields, and columns use their `@id`, ensuring unambiguous entity selection.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment in Colab or new environments)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset object
print(f"Dataset name: {metadata.name}\n{metadata.description}")

## 2. Data Overview
List available Record Sets, Fields, and Columns with their `@id` values.
This will help you reference them for subsequent extraction and processing steps.

In [ ]:
# List all record sets (tables) and their fields/columns by @id.
print("Available Record Sets and their fields:\n")

record_sets = []
for record_set in metadata.record_sets:
    print(f"- Record Set name: '{record_set.name}', @id: '{record_set.id}'")
    record_sets.append(record_set.id)
    print("  Fields/Columns:")
    for field in record_set.fields:
        col_names = [col.name for col in field.columns]
        print(f"    - Field: {field.name}, @id: {field.id}, Columns: {col_names}")
    print()
if not record_sets:
    print("No record sets found in metadata. The dataset may only have files attached or use a single record set with columns derived from files.")

# For this dataset, if no explicit record sets, try to find main files with tabular content
if not metadata.record_sets:
    print("Inspecting distributions for potential tabular datasets...")
    for d in metadata.distributions:
        print(json.dumps(d, indent=2))

## 3. Data Extraction
Load data from the main record set (or tabular file) into a DataFrame for analysis.
Use the record set and field/column `@id`s identified above.

If there are no explicit record sets, use the primary available one (the main data table) as `record_set_id`.

In [ ]:
# --- Update this value from above overview if record set(s) printed ---
# For the FAIR^2 dataset, we extract the available record set(s) by @id.

# If record_sets is empty, manually set to the main table.
if record_sets:
    # Use the first record set as the main one for downstream analysis
    main_record_set_id = record_sets[0]  # e.g., 'cr:RecordSet/proteins' etc.
else:
    # Fallback (for schemas with no explicit record set):
    # Try the default, which is 'cr:RecordSet/records' per Croissant convention, or examine files
    main_record_set_id = None

# Extract data for all listed record sets
dfs = {}
record_set_ids = record_sets if record_sets else []
if main_record_set_id:
    print(f"Loading records from record set '{main_record_set_id}' ...")
    records = list(dataset.records(record_set=main_record_set_id))
    dfs[main_record_set_id] = pd.DataFrame(records)
    print(f"Columns in main record set: {dfs[main_record_set_id].columns.tolist()}")
    display(dfs[main_record_set_id].head())
else:
    print("No explicit record set found. Trying to extract from default DataFrame...")
    records = list(dataset.records())
    if records:
        df = pd.DataFrame(records)
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
        dfs['default'] = df
    else:
        print("No tabular records present.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze a numeric field from the main record set.
We'll demonstrate filtering, normalization, and (if available) grouping.

In [ ]:
# Choose a numeric field by @id (examples: 'MW', 'PEP_COUNT', or similar, but check actual column names above)
if dfs:
    df = next(iter(dfs.values()))  # Main DataFrame
    print(f"Available columns in main DataFrame: {df.columns.tolist()}")
    
    # Find a likely numeric field, such as 'MW' (Molecular Weight), or use the first float-like field
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
    else:
        # Guess a common field like 'MW', 'mw', or 'molecular_weight', or any number columns
        for f in df.columns:
            if f.lower() in ['mw', 'molecular_weight', 'mass']:
                numeric_field = f
                break
        else:
            numeric_field = df.columns[0]  # Fallback to first column

    print(f"Using numeric field: '{numeric_field}' for analysis.")
    
    # Remove missing values
    filtered_df = df[df[numeric_field].notnull()]  # Keep only rows where the field is present
    threshold = filtered_df[numeric_field].mean()  # Example: use mean as threshold
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    display(filtered_df.head())

    # Normalize the field
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nFirst 5 normalized values for '{numeric_field}':")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Try to group by a categorical field if available (e.g., 'description', 'accession', etc.)
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping data by '{group_field}' (first 5 groups shown):")
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().head()
        display(grouped)
    else:
        print("No categorical field available for grouping.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and examine possible relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs:
    df = next(iter(dfs.values()))
    if numeric_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='cornflowerblue')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
        
        # Pair plot if another numeric column is available
        other_num = [col for col in df.columns if col != numeric_field and df[col].dtype in ['float64', 'int64']]
        if other_num:
            plt.figure(figsize=(8, 5))
            sns.scatterplot(x=df[numeric_field], y=df[other_num[0]])
            plt.xlabel(numeric_field)
            plt.ylabel(other_num[0])
            plt.title(f"Scatter plot: {numeric_field} vs {other_num[0]}")
            plt.show()
    else:
        print(f"No numeric field '{numeric_field}' to plot.")
else:
    print("No DataFrame for visualization.")

## 6. Conclusion
This notebook demonstrated how to explore the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using `mlcroissant` by accessing metadata, reviewing record sets, extracting records, conducting exploratory data analysis, and visualizing results.

Key findings:
- The dataset provides a tabular analysis of protein features from mass spectrometry on human mast cell extracellular vesicle samples.
- Common features include molecular weight and other protein descriptors that allow filtering and grouping.

Further steps might include more detailed statistical testing, cross-sample comparison, or integrating protein functional annotations based on accession identifiers.